<a href="https://colab.research.google.com/github/Rinoza-Jiffry/Langchain-Materials/blob/main/text_splitters_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import zipfile

with zipfile.ZipFile('text_splitters.zip', 'r') as zip_ref:
    zip_ref.extractall('')

# **Text Splitters** in LangChain

In [2]:
# Install the necessary packages
!pip install langchain -qU
!pip install langchain-community -qU
!pip install unstructured -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB

In [3]:
from langchain_community.document_loaders import DirectoryLoader

# Initialize the DirectoryLoader with the path to the directory for text files
loader = DirectoryLoader("/content/data", glob="**/*.txt")

# Load the text data from the directory
dataset = loader.load()

for data in dataset:
  print("------------------------")
  print(data.page_content)
  print(data.metadata)

------------------------
Agentic AI refers to a type of artificial intelligence that can act independently to achieve a specific goal, rather than simply responding to user inputs. Unlike traditional AI systems that only generate answers or follow direct instructions, agentic AI is capable of planning tasks, making decisions, and taking actions on its own.

It can break down complex problems into smaller steps, use external tools such as APIs or databases, and adjust its approach based on the results it receives. This makes it more dynamic and useful for real-world applications like virtual assistants, automated workflows, and intelligent systems that require minimal human intervention.

In essence, agentic AI goes beyond just “thinking” or “responding” by actively working toward completing tasks in a goal-oriented and autonomous manner.
{'source': '/content/data/text2.txt'}
------------------------
LangChain is a robust framework designed to facilitate the creation of applications lev

In [4]:
# Calculate the token count of each document in the dataset using a character level tokenizer
token_counts = [len(data.page_content) for data in dataset]

print(token_counts)

[824, 1784]


### Chunking the Text Using **RecursiveCharacterTextSplitter**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    length_function=len,
    separators=['\n\n', '\n', ' ', '']
)

In [6]:
# Split the text of the second document in the dataset into chunks
chunks = text_splitter.split_text(dataset[1].page_content)

len(chunks)

14

In [8]:
# Reinitialize the text splitter with a chunk size of 150 and an overlap of 20, using character length function
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    length_function=len,
    separators=['\n\n', '\n', ' ', '']
)

In [9]:
# Split the text of the second document in the dataset into chunks
chunks = text_splitter.split_text(dataset[1].page_content)

len(chunks)

15

In [11]:
chunks

['LangChain is a robust framework designed to facilitate the creation of applications leveraging large language models (LLMs). By providing a suite of',
 'a suite of tools and utilities, LangChain simplifies the process of integrating LLMs into various applications, enabling developers to build',
 'developers to build sophisticated AI-powered solutions with greater ease. Its capabilities span from natural language understanding and generation to',
 'and generation to complex data processing tasks, making it a versatile choice for developing chatbots, virtual assistants, and other AI-driven',
 'and other AI-driven applications.',
 'One of the core features of LangChain is its modular architecture, which allows developers to customize and extend the framework according to their',
 'according to their specific needs. This modularity ensures that different components, such as model training, data preprocessing, and inference, can',
 'and inference, can be independently developed and optimi

### Calculate Token Count Using tiktoken

In [10]:
# Install the tiktoken package for tokenization
!pip install tiktoken -qU

In [12]:
import tiktoken

tokenizer_model = tiktoken.encoding_for_model('gpt-3.5-turbo')
print(tokenizer_model)

<Encoding 'cl100k_base'>


In [13]:
# Get the encoding for the tokenizer
tokenizer = tiktoken.get_encoding('cl100k_base')

# Create a function to calculate the length of text in tokens using tiktoken
def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    token_length = len(tokens)
    return token_length

In [14]:
# Calculate the token count of each document in the dataset using tiktoken
token_counts = [tiktoken_len(data.page_content) for data in dataset]

print(token_counts)

[148, 286]


### Chunking the Text Using tiktoken Length Function

In [15]:
# Reinitialize the text splitter with a chunk size of 150 and an overlap of 20, using tiktoken length function
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    length_function=tiktoken_len,
    separators=['\n\n', '\n', ' ', '']
)

In [16]:
# Split the text of the second document in the dataset into chunks
chunks = text_splitter.split_text(dataset[1].page_content)

len(chunks)

3

In [17]:
chunks

['LangChain is a robust framework designed to facilitate the creation of applications leveraging large language models (LLMs). By providing a suite of tools and utilities, LangChain simplifies the process of integrating LLMs into various applications, enabling developers to build sophisticated AI-powered solutions with greater ease. Its capabilities span from natural language understanding and generation to complex data processing tasks, making it a versatile choice for developing chatbots, virtual assistants, and other AI-driven applications.',
 'One of the core features of LangChain is its modular architecture, which allows developers to customize and extend the framework according to their specific needs. This modularity ensures that different components, such as model training, data preprocessing, and inference, can be independently developed and optimized. Additionally, LangChain supports seamless integration with popular machine learning libraries and frameworks, providing a comp